## After trial and error, this should be the more successful results analysis notebook

In [26]:
import pickle
import numpy as np
import pandas as pd
from pyhere import here
from ratio_function import RatioGenerator, LogRatioGenerator
import itertools
from plotnine import *
from sklearn.model_selection import train_test_split
from sklearn import set_config
from skopt import BayesSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from skopt.space import Categorical, Real, Integer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score, cross_validate, KFold
from sklearn.metrics import root_mean_squared_error, r2_score

set_config(transform_output="pandas")

Loading all the reuquired stuff

In [27]:
with open(here("pipeline/spaces", "space_list.pkl"), "rb") as file:
    search_space_list = pickle.load(file)

with open(here("pipeline/spaces", "model_list.pkl"), "rb") as file2:
    model_list = pickle.load(file2)

phot = pd.read_csv(here("data/cleaned", "MIRION_cleaned_everything.csv"))

flux_cols = ['F1100', 'F870', 'F500', 'F350', 'F250', 'F160', 'F70', 'F24', 'F12', 'F8']

model_pipe = Pipeline([
        ('impute', SimpleImputer()),
        ('ratio', RatioGenerator(cols=flux_cols)),
        ('scale', RobustScaler()),
        ('model', 'passthrough')
    ])

remove_properties = ['LRATIO', 'T_BOL', 'LM', 'L_BOL', 'MASS', 'DIAM', 'SURF_DENS', 'YB', 'TEMP']

# Bolometric luminosity

In [28]:
with open(here("pipeline/results", "L_BOL_results.pkl"), 'rb') as file:
    lbol_results = pickle.load(file)

print(lbol_results['cat_log_CV'])
print(lbol_results['cat_log_params'])

0.4231401644192835
OrderedDict({'impute': 'passthrough', 'model__bagging_temperature': 0.0, 'model__border_count': 255, 'model__colsample_bylevel': 0.43274901277331335, 'model__depth': 5, 'model__grow_policy': 'SymmetricTree', 'model__iterations': 2659, 'model__l2_leaf_reg': 0.001, 'model__learning_rate': 0.02371580651393871, 'model__min_data_in_leaf': 1, 'model__random_strength': 10.0, 'ratio': RatioGenerator(cols=['F8', 'F12', 'F24', 'F70', 'F160', 'F250', 'F350', 'F500',
                     'F870', 'F1100']), 'scale': StandardScaler()})


In [ ]:
y_log_lbol = np.log(phot['L_BOL'])
X_lbol = phot.drop(columns=remove_properties)

lbol_X_train, lbol_X_test, lbol_y_train, lbol_y_test = train_test_split(X_lbol, y_log_lbol, test_size = 0.2, random_state=2026)

best_lbol_model = model_pipe.set_params(model=model_list[0])
best_lbol_model.set_params(**lbol_results['cat_log_params'])

best_lbol_model.fit(lbol_X_train, lbol_y_train)
lbol_y_preds = best_lbol_model.predict(lbol_X_test)
lbol_rmse = root_mean_squared_error(lbol_y_test, lbol_y_preds)
lbol_r2 = r2_score(lbol_y_test, lbol_y_preds)
print(f"Final training set RMSE (log scale) for Bolometric luminosity is {lbol_rmse:.4f}, and r squared is {lbol_r2:.4f}.")

Final training set RMSE for Bolometric luminosity is 0.4171, and r squared is 0.9667
